In [2]:
import pandas as pd

df_ground_truth = pd.read_csv("../data/ground_truth.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

In [3]:
ground_truth[10]

{'question': 'Where can I find the direct Zoom link to join the live workshop sessions?',
 'document': '489dd1c9d9'}

In [4]:
from ingest import load_faq_data, build_index

documents = load_faq_data()

documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

documents = documents_llm
index = build_index(documents)

In [5]:
doc_idx = {}

for doc in documents:
    doc_idx[doc["id"]] = doc

In [6]:
q = ground_truth[10]
q

{'question': 'Where can I find the direct Zoom link to join the live workshop sessions?',
 'document': '489dd1c9d9'}

In [7]:
doc_idx[q['document']]

{'id': '489dd1c9d9',
 'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'What is the video/zoom link to the stream for the “Office Hours” or live/workshop sessions?',
 'answer': 'The zoom link is only published to instructors/presenters/TAs.\n\nStudents participate via YouTube Live and submit questions to Slido (link is pinned in the chat when live). The video URL should be posted in the [announcements channel on Telegram and Slack](https://t.me/dezoomcamp) before it begins. You can also watch live on the DataTalksClub [YouTube Channel](https://www.youtube.com/c/DataTalksClub).\n\nDon’t post questions in chat as they may be missed if the room is very active.'}

In [8]:
from dotenv import load_dotenv
load_dotenv()
from google import genai
gemini_client = genai.Client()

In [9]:
from evaluation_utils import RAGWithUsage

assistant = RAGWithUsage(
    index=index,
    llm_client=gemini_client,
    course='llm-zoomcamp',
)

In [10]:
q['question']

'Where can I find the direct Zoom link to join the live workshop sessions?'

In [11]:
answer = assistant.rag(q['question'])

In [12]:
assistant.total_cost()

9.15e-05

In [13]:
print(answer)

I don't know based on the provided context.


In [14]:
doc_id = q["document"]
original_doc = doc_idx[doc_id]
answer_orig = original_doc["answer"]

answer_orig

'The zoom link is only published to instructors/presenters/TAs.\n\nStudents participate via YouTube Live and submit questions to Slido (link is pinned in the chat when live). The video URL should be posted in the [announcements channel on Telegram and Slack](https://t.me/dezoomcamp) before it begins. You can also watch live on the DataTalksClub [YouTube Channel](https://www.youtube.com/c/DataTalksClub).\n\nDon’t post questions in chat as they may be missed if the room is very active.'

In [15]:
rag_result = {
    "question": q['question'],
    "answer_llm": answer,
    "answer_orig": answer_orig,
    "document": doc_id,
}

rag_result

{'question': 'Where can I find the direct Zoom link to join the live workshop sessions?',
 'answer_llm': "I don't know based on the provided context.",
 'answer_orig': 'The zoom link is only published to instructors/presenters/TAs.\n\nStudents participate via YouTube Live and submit questions to Slido (link is pinned in the chat when live). The video URL should be posted in the [announcements channel on Telegram and Slack](https://t.me/dezoomcamp) before it begins. You can also watch live on the DataTalksClub [YouTube Channel](https://www.youtube.com/c/DataTalksClub).\n\nDon’t post questions in chat as they may be missed if the room is very active.',
 'document': '489dd1c9d9'}

In [16]:
def generate_rag_answer(rec):
    question = rec["question"]
    doc_id = rec["document"]
    original_doc = doc_idx[doc_id]

    answer_llm = assistant.rag(question)
    answer_orig = original_doc["answer"]

    result = {
        "question": question,
        "answer_llm": answer_llm,
        "answer_orig": answer_orig,
        "document": doc_id,
    }

    return result

In [17]:
record = generate_rag_answer(q)
record

{'question': 'Where can I find the direct Zoom link to join the live workshop sessions?',
 'answer_llm': 'The zoom link is only published to instructors/presenters/TAs.',
 'answer_orig': 'The zoom link is only published to instructors/presenters/TAs.\n\nStudents participate via YouTube Live and submit questions to Slido (link is pinned in the chat when live). The video URL should be posted in the [announcements channel on Telegram and Slack](https://t.me/dezoomcamp) before it begins. You can also watch live on the DataTalksClub [YouTube Channel](https://www.youtube.com/c/DataTalksClub).\n\nDon’t post questions in chat as they may be missed if the room is very active.',
 'document': '489dd1c9d9'}

In [18]:
assistant.total_cost()

0.0001846

In [19]:
assistant.reset_usage()

In [20]:
assistant.total_cost()

0.0

In [ ]:
import time
from tqdm.auto import tqdm


results = []
for i, rec in tqdm(enumerate(ground_truth), total=len(ground_truth)):
    results.append(generate_rag_answer(rec))
    if i < len(ground_truth) - 1:
        time.sleep(4.5)

In [ ]:
import json
with open("results_complete.json", "w") as f:
    json.dump(results, f, indent=2)

In [25]:
start = len(results)

for i, rec in tqdm(
    enumerate(ground_truth[start:], start=start),
    total=len(ground_truth) - start
):
    results.append(generate_rag_answer(rec))

    if i < len(ground_truth) - 1:
        time.sleep(4.5)

0it [00:00, ?it/s]


In [26]:
df_results = pd.DataFrame(results)

In [27]:
df_results.head()

,question,answer_llm,answer_orig,document
0,Is it too late to enroll since I just found ou...,You can also just start learning and submittin...,"Yes, but if you want to receive a certificate,...",74eb249bbf
1,Can I catch up and start participating even th...,"Yes, you can start whenever you want. The vide...","Yes, but if you want to receive a certificate,...",74eb249bbf
2,"If I join now, will I still be eligible to ear...","Yes, but you need to submit your project while...","Yes, but if you want to receive a certificate,...",74eb249bbf
3,Are late sign-ups allowed for students wanting...,I don't know based on the provided context.,"Yes, but if you want to receive a certificate,...",74eb249bbf
4,What is the deal with submitting my final proj...,I don't know based on the provided context.,"Yes, but if you want to receive a certificate,...",74eb249bbf


In [28]:
assistant.total_cost()

0.0125562

In [36]:
df_results.to_csv("../data/rag-answers-new.csv", index=False)